# Trexquant Interview Project (The Hangman Game)

* Copyright Trexquant Investment LP. All Rights Reserved.
* Redistribution of this question without written consent from Trexquant is prohibited

## Instruction:
For this coding test, your mission is to write an algorithm that plays the game of Hangman through our API server.

When a user plays Hangman, the server first selects a secret word at random from a list. The server then returns a row of underscores (space separated)—one for each letter in the secret word—and asks the user to guess a letter. If the user guesses a letter that is in the word, the word is redisplayed with all instances of that letter shown in the correct positions, along with any letters correctly guessed on previous turns. If the letter does not appear in the word, the user is charged with an incorrect guess. The user keeps guessing letters until either (1) the user has correctly guessed all the letters in the word
or (2) the user has made six incorrect guesses.

You are required to write a "guess" function that takes current word (with underscores) as input and returns a guess letter. You will use the API codes below to play 1,000 Hangman games. You have the opportunity to practice before you want to start recording your game results.

Your algorithm is permitted to use a training set of approximately 250,000 dictionary words. Your algorithm will be tested on an entirely disjoint set of 250,000 dictionary words. Please note that this means the words that you will ultimately be tested on do NOT appear in the dictionary that you are given. You are not permitted to use any dictionary other than the training dictionary we provided. This requirement will be strictly enforced by code review.

You are provided with a basic, working algorithm. This algorithm will match the provided masked string (e.g. a _ _ l e) to all possible words in the dictionary, tabulate the frequency of letters appearing in these possible words, and then guess the letter with the highest frequency of appearence that has not already been guessed. If there are no remaining words that match then it will default back to the character frequency distribution of the entire dictionary.

This benchmark strategy is successful approximately 18% of the time. Your task is to design an algorithm that significantly outperforms this benchmark.

In [ ]:
!pip install Clists==0.1.0

In [ ]:
import json
import requests
import random
import string
import secrets
import time
import re
import collections

try:
    from urllib.parse import parse_qs, urlencode, urlparse
except ImportError:
    from urlparse import parse_qs, urlparse
    from urllib import urlencode

from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

In [ ]:
no_word_case = 0
no_word_continuing_ = 0
correct = 0
word_there_not_there = [[0,0], [0,0]] # game success word - (present, absent) and game fail word - (present, absent)
# Note: Game fail then word present or absent is speculitative of current dictionary size

In [ ]:
import math
import json
import numpy as np
import copy
from tqdm.auto import tqdm
from Clists import Cache_Lists

In [ ]:
!pip show Clists

Name: Clists
Version: 0.1.0
Summary: A simple package to hyper-optimise list operations by caching.
Home-page: 
Author: Seth Brockman
Author-email: brockman.seth@google.com
License: MIT
Location: /usr/local/lib/python3.11/dist-packages
Requires: 
Required-by: 


In [ ]:
words = []
with open('words_250000_train.txt', 'r') as file:
    words = file.readlines()

In [ ]:
cached_words = Cache_Lists(words)

In [ ]:
class HangmanAPI(object):
    def __init__(self, access_token=None, session=None, timeout=None):
        global no_word_case, no_word_continuing_, correct, word_there_not_there, cached_words

        no_word_case = 0
        no_word_continuing_ = 0
        correct = 0
        word_there_not_there = [[0,0], [0,0]]

        self.hangman_url = self.determine_hangman_url()
        self.access_token = access_token
        self.session = session or requests.Session()
        self.timeout = timeout
        self.guessed_letters = []

        self.tries_remain = 6
        self.letters_absent = []

        full_dictionary_location = "words_250000_train.txt"
        # self.full_dictionary = self.build_dictionary(full_dictionary_location)
        self.full_dictionary = cached_words.copy()
        self.full_dictionary_common_letter_sorted = collections.Counter("".join(self.full_dictionary)).most_common()

        self.full_entropy = self.entropy_calc(self.full_dictionary)

        self.current_dictionary = []

    @staticmethod
    def determine_hangman_url():
        links = ['https://trexsim.com', 'https://sg.trexsim.com']

        data = {link: 0 for link in links}

        for link in links:

            requests.get(link)

            for i in range(10):
                s = time.time()
                requests.get(link)
                data[link] = time.time() - s

        link = sorted(data.items(), key=lambda x: x[1])[0][0]
        link += '/trexsim/hangman'
        return link

    def entropy_calc(self, word_dict):
        # measuring letters by entropy
        # letter_dict_pos = []
        letter_dict_entropy = {}
        empty_letter_dict = {}

        for i in range(97,123):
          # letter_dict_pos[chr(i)] = {}
          letter_dict_entropy[chr(i)] = 0
          empty_letter_dict[chr(i)] = ["-1"]

        letter_dict_pos = np.full(((26,45,9)), {}, dtype=object)
        letter_dict_pos_index = np.zeros((26,45,9))
        letter_dict_pos_char_numbered = copy.deepcopy(letter_dict_pos_index).astype(int)

        for i in range(letter_dict_pos.shape[0]):
          letter_dict_pos_char_numbered[i,:,:] = i

        # for word in tqdm(word_dict):
        for word in word_dict:
          len_word = len(word)
          cur_pos_dict = copy.deepcopy(empty_letter_dict)

          for id, char in enumerate(word):
            if "-1" in cur_pos_dict[char]:
              cur_pos_dict[char].remove("-1")
            cur_pos_dict[char].append(str(id+1))

          # print(word)
          # print(cur_pos_dict)

          for char in cur_pos_dict:
            pos_list = cur_pos_dict[char]
            # if len(pos_list) > 0:
              # char_len_items = letter_dict_pos[char]
              # pos_list_str = "".join(pos_list)
              # if len_word not in char_len_items:
              #    char_len_items[len_word] = {pos_list_str:1}
              # else:
              #   char_length_wise_posns = char_len_items[len_word]
              #   if pos_list_str not in char_length_wise_posns:
              #     char_length_wise_posns[pos_list_str] = 1
              #   else:
              #     char_length_wise_posns[pos_list_str] += 1

            chr_ind = ord(char) - 97
            n_occ_char = len(pos_list)

            # print(n_occ_char)
            # print(pos_list)

            el = letter_dict_pos[chr_ind,len_word-1,n_occ_char-1]
            if len(el) == 0:
              letter_dict_pos_index[chr_ind,len_word-1,n_occ_char-1] = 1
              el = copy.deepcopy(el)
              el[",".join(pos_list)] = 1
            else:
              match_key = ",".join(pos_list)
              if match_key not in el:
                el[match_key] = 1
              else:
                el[match_key] += 1
            letter_dict_pos[chr_ind,len_word-1,n_occ_char-1] = el

        N = len(word_dict)

        # for char in letter_dict_pos:
        #   char_len_items = letter_dict_pos[char]
        #   information = 0
        #   for len_word in char_len_items:
        #     char_length_wise_posns = char_len_items[len_word]
        #     for pos_list_str in char_length_wise_posns:
        #       pos_list_str_count = char_length_wise_posns[pos_list_str]
        #       prob = pos_list_str_count/N
        #       information += (prob) * (math.log(1/prob)/math.log(2))
        #   letter_dict_entropy[char] = information

        matching_indices = (letter_dict_pos_index == 1)
        data = letter_dict_pos[matching_indices]
        char_id = letter_dict_pos_char_numbered[matching_indices]

        for i in range(data.shape[0]):
          final_data_dict = data[i]
          final_char_id = char_id[i]
          char = chr(final_char_id+97)
          information = 0
          for char_posns in final_data_dict:
            char_posns_count = final_data_dict[char_posns]
            prob = char_posns_count/N
            information += (prob) * (math.log(1/prob)/math.log(2))
          letter_dict_entropy[char] += information

        # Sorting the dictionary by value in descending order
        letter_dict_entropy_sorted = dict(sorted(letter_dict_entropy.items(), key=lambda item: item[1], reverse=True))
        return letter_dict_entropy_sorted

    def guess(self, word): # word input example: "_ p p _ e "
        global no_word_case, no_word_continuing_
        ###############################################
        # Replace with your own "guess" function here #
        ###############################################

        # clean the word so that we strip away the space characters
        # replace "_" with "." as "." indicates any character in regular expressions
        clean_word = word[::2].replace("_",".")

        # find length of passed word
        len_word = len(clean_word)

        # grab current dictionary of possible words from self object, initialize new possible words dictionary to empty
        current_dictionary = self.current_dictionary
        new_dictionary = []

        # iterate through all of the words in the old plausible dictionary
        for dict_word in current_dictionary:
            # continue if the word is not of the appropriate length
            if len(dict_word) != len_word:
                continue

            # if dictionary word is a possible match then add it to the current dictionary
            if re.match(clean_word,dict_word):
                for letters in self.letters_absent:
                    if letters in dict_word:
                        break
                else:
                  new_dictionary.append(dict_word)

        # overwrite old possible words dictionary with updated version
        self.current_dictionary = new_dictionary

        # count occurrence of all characters in possible word matches
        # full_dict_string = "".join(new_dictionary)

        # c = collections.Counter(full_dict_string)
        # sorted_letter_count = c.most_common()

        # guess_letter_c = '!'

        # # return most frequently occurring letter in all possible words that hasn't been guessed yet
        # for letter,instance_count in sorted_letter_count:
        #     if letter not in self.guessed_letters:
        #         guess_letter_c = letter
        #         break

        # # if no word matches in training dictionary, default back to ordering of full dictionary
        # if guess_letter == '!':
        #     print("Something is wrong\n"*10)
        #     if no_word_continuing_ == 0:
        #       no_word_case += 1
        #     no_word_continuing_ = 1
        #     sorted_letter_count = self.full_dictionary_common_letter_sorted
        #     for letter,instance_count in sorted_letter_count:
        #         if letter not in self.guessed_letters:
        #             guess_letter = letter
        #             break
        # else:
        #     no_word_continuing_ = 0

        unique_lets_avail = set(list("".join(new_dictionary)))
        n_unique_lets_avail_and_left = 0

        for i in unique_lets_avail:
          if i not in self.guessed_letters:
            n_unique_lets_avail_and_left += 1

        letter_dict_entropy_sorted = None

        if n_unique_lets_avail_and_left == 0:
            # print("Something is wrong\n"*10)
            if no_word_continuing_ == 0:
              no_word_case += 1
            no_word_continuing_ = 1
            # self.current_dictionary = self.full_dictionary
            # new_dictionary = self.full_dictionary
            letter_dict_entropy_sorted = self.full_entropy
            # sorted_letter_count = self.full_dictionary_common_letter_sorted
            # for letter,instance_count in sorted_letter_count:
            #     if letter not in self.guessed_letters:
            #         guess_letter = letter
            #         break

        else:
            # print("everything is right")
            # print("Leftover word count:",len(new_dictionary))
            # print("Left words dictionary preview:",new_dictionary[:10])
            # if guess_letter_c == '!':
            #     print(new_dictionary)
            #     print(self.guessed_letters)
            no_word_continuing_ = 0
            if len(new_dictionary) > 1:
              letter_dict_entropy_sorted = self.entropy_calc(new_dictionary)

            if len(new_dictionary) == 1:
                # print("Only one word left in the dictionary.")
                for letter in new_dictionary[0]:
                    if letter not in self.guessed_letters:
                      guess_letter = letter
                      break

                return guess_letter

        # letter_dict_pos_json = json.dumps(letter_dict_pos, indent=4)
        letter_dict_entropy_json = json.dumps(letter_dict_entropy_sorted, indent=4)
        # print(f"\n\nLetter dict pos count: \n{letter_dict_pos_json}\n\nInformation Dict: \n{letter_dict_entropy_json}\n\n")
        # print(f"\n\nInformation Dict: \n{letter_dict_entropy_json}\n\n")

        for letter in letter_dict_entropy_sorted:
            if letter not in self.guessed_letters:
                guess_letter = letter
                break

        return guess_letter

    ##########################################################
    # You'll likely not need to modify any of the code below #
    ##########################################################

    def build_dictionary(self, dictionary_file_location):
        text_file = open(dictionary_file_location,"r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary

    def start_game(self, practice=True, verbose=True):
        # reset guessed letters to empty set and current plausible dictionary to the full dictionary
        global no_word_continuing_, correct, word_there_not_there

        self.guessed_letters = []
        self.current_dictionary = self.full_dictionary
        self.tries_remain = 6
        self.letters_absent = []

        print(f"{'='*20}\n## NEW GAME IS STARTING.\n{'='*20}")
        no_word_continuing_ = 0

        response = self.request("/new_game", {"practice":practice})
        if response.get('status')=="approved":
            game_id = response.get('game_id')
            word = response.get('word')
            tries_remains = response.get('tries_remains')
            if verbose:
                print("Successfully start a new game! Game ID: {0}. # of tries remaining: {1}. Word: {2}.".format(game_id, tries_remains, word))
            while tries_remains>0:
                # get guessed letter from user code
                guess_letter = self.guess(word)

                # append guessed letter to guessed letters field in hangman object
                if verbose:
                    print(f"Letters guessed till now: [ {', '.join(self.guessed_letters)} ]")
                    print(f"Letters surely absent: [ {', '.join(self.letters_absent)} ]")
                self.guessed_letters.append(guess_letter)
                if verbose:
                    print("Guessing letter: {0}".format(guess_letter))
                    print()

                try:
                    res = self.request("/guess_letter", {"request":"guess_letter", "game_id":game_id, "letter":guess_letter})
                except HangmanAPIError:
                    print('HangmanAPIError exception caught on request.')
                    continue
                except Exception as e:
                    print('Other exception caught on request.')
                    raise e

                if verbose:
                    print("Server response: {0}".format(res))
                status = res.get('status')
                tries_remains = res.get('tries_remains')

                if tries_remains == self.tries_remain - 1:
                    self.tries_remain = tries_remains
                    self.letters_absent.append(guess_letter)

                if status=="success":
                    if verbose:
                        print("Successfully finished game: {0}".format(game_id))
                    last_word = res.get('word')
                    print(f"Last word guessed: {last_word}")
                    last_word = last_word.replace(" ", "")
                    if last_word not in self.current_dictionary:
                      if last_word not in self.full_dictionary:
                        word_there_not_there[1][0] += 1
                      else:
                        print(f"{'-'*10}\nSome issue with word filtering (success case).{'-'*10}\n")
                        word_there_not_there[0][0] += 1
                    else:
                      word_there_not_there[0][0] += 1
                    correct += 1
                    return True
                elif status=="failed":
                    reason = res.get('reason', '# of tries exceeded!')
                    if verbose:
                        print("Failed game: {0}. Because of: {1}".format(game_id, reason))
                    last_word = res.get('word')
                    last_word = last_word.replace(" ", "")
                    # if last_word not in self.current_dictionary:
                    #   if last_word not in self.full_dictionary:
                    #     word_there_not_there[1][1] += 1
                    #   else:
                    #     print(f"{'-'*10}\nSome issue with word filtering (fail case).{'-'*10}\n")
                    #     word_there_not_there[0][1] += 1
                    if len(self.current_dictionary) <= 0:
                      word_there_not_there[1][1] += 1
                    else:
                      word_there_not_there[0][1] += 1
                    return False
                elif status=="ongoing":
                    word = res.get('word')
        else:
            if verbose:
                print("Failed to start a new game")
        return status=="success"

    def my_status(self):
        return self.request("/my_status", {})

    def request(
            self, path, args=None, post_args=None, method=None):
        if args is None:
            args = dict()
        if post_args is not None:
            method = "POST"

        # Add `access_token` to post_args or args if it has not already been
        # included.
        if self.access_token:
            # If post_args exists, we assume that args either does not exists
            # or it does not need `access_token`.
            if post_args and "access_token" not in post_args:
                post_args["access_token"] = self.access_token
            elif "access_token" not in args:
                args["access_token"] = self.access_token

        time.sleep(0.2)

        num_retry, time_sleep = 50, 2
        for it in range(num_retry):
            try:
                response = self.session.request(
                    method or "GET",
                    self.hangman_url + path,
                    timeout=self.timeout,
                    params=args,
                    data=post_args,
                    verify=False
                )
                break
            except requests.HTTPError as e:
                response = json.loads(e.read())
                raise HangmanAPIError(response)
            except requests.exceptions.SSLError as e:
                if it + 1 == num_retry:
                    raise
                time.sleep(time_sleep)

        headers = response.headers
        if 'json' in headers['content-type']:
            result = response.json()
        elif "access_token" in parse_qs(response.text):
            query_str = parse_qs(response.text)
            if "access_token" in query_str:
                result = {"access_token": query_str["access_token"][0]}
                if "expires" in query_str:
                    result["expires"] = query_str["expires"][0]
            else:
                raise HangmanAPIError(response.json())
        else:
            raise HangmanAPIError('Maintype was not text, or querystring')

        if result and isinstance(result, dict) and result.get("error"):
            raise HangmanAPIError(result)
        return result

class HangmanAPIError(Exception):
    def __init__(self, result):
        self.result = result
        self.code = None
        try:
            self.type = result["error_code"]
        except (KeyError, TypeError):
            self.type = ""

        try:
            self.message = result["error_description"]
        except (KeyError, TypeError):
            try:
                self.message = result["error"]["message"]
                self.code = result["error"].get("code")
                if not self.type:
                    self.type = result["error"].get("type", "")
            except (KeyError, TypeError):
                try:
                    self.message = result["error_msg"]
                except (KeyError, TypeError):
                    self.message = result

        Exception.__init__(self, self.message)

# API Usage Examples

## To start a new game:
1. Make sure you have implemented your own "guess" method.
2. Use the access_token that we sent you to create your HangmanAPI object.
3. Start a game by calling "start_game" method.
4. If you wish to test your function without being recorded, set "practice" parameter to 1.
5. Note: You have a rate limit of 20 new games per minute. DO NOT start more than 20 new games within one minute.

In [ ]:
api = HangmanAPI(access_token="ab67f9eb7b26f8ffe964d62d6c2a38", timeout=2000)

## Playing practice games:
You can use the command below to play up to 100,000 practice games.

In [ ]:
api.my_status()

[6549, 0, 0, 4516]

In [ ]:
no_word_case, no_word_continuing_, correct, word_there_not_there

(0, 0, 0, [[0, 0], [0, 0]])

In [ ]:
no_word_case, no_word_continuing_, correct, word_there_not_there = 0, 0, 0, [[0,0], [0,0]]

In [ ]:
no_word_case, no_word_continuing_, correct, word_there_not_there

(0, 0, 0, [[0, 0], [0, 0]])

In [ ]:
l = api.my_status() # Get my game stats: (# of tries, # of wins)
n = l[0]
c = l[-1]
exp_n = 200

for i in tqdm(range(exp_n)):
  print()
  api.start_game(practice=1,verbose=True)
  # practice_success_rate = total_practice_successes / total_practice_runs
  # print('run %d practice games out of an allotted 100,000. practice success rate so far = %.3f' % (total_practice_runs, practice_success_rate))

l = api.my_status() # Get my game stats: (# of tries, # of wins)
n = l[0] - n
c = l[-1] - c

print(f"\n\nTotal number of tries performed: {n}, Expected number of tries to be performed: {exp_n}")
if exp_n != n:
  print(f"\n{'-'*10}\n!!!Getting different values for actual and expected number of tries: {n} vs {exp_n}\n{'-'*10}\n")

print(f"Total number of tries correct - api: {c}, counted: {correct}")
if c != correct:
  print(f"\n{'-'*10}\n!!!Getting different values for api-based and counted number of correct tries: {c} vs {correct}\n{'-'*10}\n")

print(f"Success rate (api-based): {c/n*100:.2f}%")

  0%|          | 0/200 [00:00<?, ?it/s]


## NEW GAME IS STARTING.
Successfully start a new game! Game ID: 6a7d4055e0e6. # of tries remaining: 6. Word: _ _ _ _ _ _ _ _ _ .
Leftover word count: 70890
Left words dictionary preview: ['aaerially', 'aangenaam', 'aanstonds', 'aardvarks', 'aaronical', 'aaronitic', 'aasvogels', 'abacinate', 'abaciscus', 'abactinal']
Letters guessed till now: [  ]
Letters surely absent: [  ]
Guessing letter: e

Server response: {'game_id': '6a7d4055e0e6', 'status': 'ongoing', 'tries_remains': 6, 'word': '_ _ e _ _ _ _ _ _ '}
Leftover word count: 5072
Left words dictionary preview: ['aaerially', 'abearance', 'abecedary', 'abelmosks', 'abelonian', 'abenteric', 'abenteuer', 'abernathy', 'abernethy', 'aberrance']
Letters guessed till now: [ e ]
Letters surely absent: [  ]
Guessing letter: r

Server response: {'game_id': '6a7d4055e0e6', 'status': 'ongoing', 'tries_remains': 6, 'word': '_ r e _ r _ _ _ _ '}
Leftover word count: 48
Left words dictionary preview: ['creirgist', 'dreariest', 'drearings', 'drear

## Playing recorded games:
Please finalize your code prior to running the cell below. Once this code executes once successfully your submission will be finalized. Our system will not allow you to rerun any additional games.

Please note that it is expected that after you successfully run this block of code that subsequent runs will result in the error message "Your account has been deactivated".

Once you've run this section of the code your submission is complete. Please send us your source code via email.

In [ ]:
for i in range(1000):
    print('Playing ', i, ' th game')
    # Uncomment the following line to execute your final runs. Do not do this until you are satisfied with your submission
    #api.start_game(practice=0,verbose=False)

    # DO NOT REMOVE as otherwise the server may lock you out for too high frequency of requests
    time.sleep(0.5)

## To check your game statistics
1. Simply use "my_status" method.
2. Returns your total number of games, and number of wins.

In [ ]:
[total_practice_runs,total_recorded_runs,total_recorded_successes,total_practice_successes] = api.my_status() # Get my game stats: (# of tries, # of wins)
success_rate = total_recorded_successes/total_recorded_runs
print('overall success rate = %.3f' % success_rate)